# Experiments 87 -  ***OPTUNA SEARCH***
Pruebas de validación ajustando Confidence y IOU setup para Non-Maximum Suppression (NMS).

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Weights:** Exp. 86 *(Full fine-tuned, no freeze)*
- **Experiments:**
    1. Optuna hyperparam search: `conf=0.15/0.50` | `iou=0.3/0.6`
- **Reference:** Default parameters: `conf=0.25` | `iou=0.6`

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 14.5 MB/s eta 0:00:00


In [24]:
!pip install ultralytics

## Helper Functions

In [105]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [106]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [107]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [108]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [109]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [122]:
import os
import shutil

def save_on_cloud(source: str, destination: str):
    """
    Saves a folder or a file to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder or file.
        destination (str): The path to the destination folder or the destination path for the file (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source exists
        if not os.path.exists(source):
            print(f"❌ Source path not found: {source}")
            return

        # Determine if the source is a file or a directory
        if os.path.isfile(source):
            # If source is a file, copy it directly
            # Ensure the destination directory exists
            dest_dir = os.path.dirname(destination)
            if dest_dir and not os.path.exists(dest_dir):
                os.makedirs(dest_dir)

            shutil.copy(source, destination)
            print("✅ File copied successfully:\n  ", source, "\n  -->", destination)

        elif os.path.isdir(source):
            # If source is a directory, copy the entire tree
            # Ensure the destination directory exists (this is handled by copytree with dirs_exist_ok=True, but good to be explicit)
            if not os.path.exists(destination):
                 os.makedirs(destination)

            shutil.copytree(source, destination, dirs_exist_ok=True)
            print("✅ Folder copied successfully:\n  ", source, "\n  -->", destination)
        else:
            print(f"❌ Source path is neither a file nor a directory: {source}")


    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Graph functions

In [111]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [112]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f2:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

    return accuracy, precision, recall, f1, f2, fm

In [113]:
# def ref_metric(precision,recall):
#     return 1.25 * (precision * recall) / (0.25 * precision + recall)

In [114]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [115]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("\n✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"\n❌ An error occurred: {e}")

  #return json_data


In [116]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  #print("Total objects detected:", total_det)
  #print("Confusion matrix:")
  #for row in percentages:
  #    print(row)

  return matrix


In [117]:
def numoji(numero):
  """
  Convierte un número entero del 1 al 10 a su emoji correspondiente.

  Args:
    numero: Un entero entre 1 y 10.

  Returns:
    Un string con el emoji correspondiente al número, o "0️⃣" si el número
    está fuera del rango.
  """
  if 0 <= numero <= 10:
    emoji_map = {
        0: "0️⃣",
        1: "1️⃣",
        2: "2️⃣",
        3: "3️⃣",
        4: "4️⃣",
        5: "5️⃣",
        6: "6️⃣",
        7: "7️⃣",
        8: "8️⃣",
        9: "9️⃣",
        10: "🔟"
    }
    return emoji_map[numero]
  else:
    return "*️⃣"

# Datasets builder

## Importing from Drive

In [17]:
!rm -rf /content/sample_data

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8.640px.aug.v1
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  3.5m.v4i.yolov8_blended.640px.aug.v1
3.5m.v3i.yolov8.640px_clahe	       3.5m.v5i.yolov8.640px-2steps.aug2
3.5m.v3i.yolov8.640px.soil_aug	       Inference
3.5m.v4i.yolov8.640px		       models
3.5m.v4i.yolov8.640px_209	       optuna_yolov8_f1_study.db


In [32]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 14 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8.640px_209',
 '3.5m.v4i.yolov8_blended.640px.aug.v1',
 '3.5m.v4i.yolov8.640px.aug.v1',
 '3.5m.v5i.yolov8.640px-2steps.aug2']

In [42]:
choose_dataset = 14
index = choose_dataset - 1
dataset_name = os.listdir(drive_path)[index]
print("Chosen dataset:", dataset_name)

Chosen dataset: 3.5m.v5i.yolov8.640px-2steps.aug2


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [22]:
# Option 2 (download just the needed dataset)
#cloud_path = f"{drive_path}/{dataset_name}/"
#local_path = f"/content/YOLO/"

In [46]:
# Option 3 (download just what's needed)
set_split = 'valid'
cloud_path = f"{drive_path}/{dataset_name}/{set_split}/"
yaml_path = f"{drive_path}/{dataset_name}/data.yaml"
local_path = f"/content/YOLO/{dataset_name}/"

In [40]:
!mkdir $local_path
!cp -r $cloud_path $local_path

In [48]:
!cp $yaml_path $local_path

In [79]:
src_folder = f"/content/YOLO/{dataset_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml'

---

In [58]:
models_path = f'{drive_path}/models'
drive_models_path = os.listdir(models_path)
drive_models = len(drive_models_path)
if (drive_models) > 1:
    print("There are %d dataset options:" % drive_models)
else:
    print("Theres is only 1 dataset:")
drive_models_path

There are 4 dataset options:


['best_e26.pt', 'best_e50.pt', 'best_e86.pt', 'best_e79.pt']

In [55]:
choose_model = 3
index = choose_model - 1
model_name = os.listdir(models_path)[index]
print("Chosen model:", model_name)

Chosen model: best_e86.pt


In [63]:
# Option 2 (download just the dataset needed)
model_cloud_path = f"{models_path}/{model_name}"
model_local_path = f"/content/YOLO/"
!cp -r $model_cloud_path $model_local_path
model_weights = f"/content/YOLO/{model_name}"

In [99]:
import re
match = re.search(r"e(\d+)\.", model_name)

if match:
    model_num = match.group(1)
    model_num = f"e{model_num}"
    print(model_num)
else:
    print("No se encontró el número en el nombre del archivo.")

e86


## Download model

In [66]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [67]:
# Load stored model (Exp. 26)
#model_weights = "/content/drive/MyDrive/YOLO/models/best_e26.pt"
model = YOLO(model_weights)

# Experiment

### Training optimization

In [126]:
# Libera memoria de la GPU en caso de OOM error
import torch
import gc

for i in range(20):
  torch.cuda.empty_cache()
  gc.collect()

In [81]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [82]:
!nvidia-smi

Thu May 15 18:11:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             30W /   70W |     214MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [83]:
!yolo version

/bin/bash: line 1: yolo: command not found


-----
## Optuna search
### *Full fine-tuned (no freeze) | Hyperparameters serach*
Applying optuna for finding values of `conf` & `iou` that maximize F1/2-score.

In [131]:
gpu_limit = 0.5

In [100]:
average_time = 17 # average time per optuna experiment measured earlier

### Validation

In [132]:
import optuna

def objective(trial):

    # Suggest values for conf and iou
    conf_threshold = trial.suggest_float("conf", 0.15, 0.3) # Define a reasonable range
    iou_threshold = trial.suggest_float("iou", 0.3, 0.6)   # Define a reasonable range
    emoji_number = "".join([numoji(int(i)) for i in str(trial.number)])
    print(f"\n\n{emoji_number} Trial {trial.number}: Trying conf={conf_threshold:.4f}, iou={iou_threshold:.4f}")

    try:
        # Run validation with the suggested hyperparameters
        # Disable save_json unless you really need the files,
        # to avoid filling up the disk during optimization.
        # verbose=False to reduce output during Optuna trials,
        # unless you need to debug each trial.

        # 1. EMPTY MEMORY
        for i in range(20):
            torch.cuda.empty_cache()
            gc.collect()

        # 2. VALIDATION
        print("VALIDATION:")
        results = model.val(
            data=data,
            batch=64,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=True, # Keep verbose to see detailed output
            save_json=True # Save JSON
        )

        # 3. CONFIRMATION
        # Extract the Accuracy & F½-Score.
        if hasattr(results, 'results_dict') and results.results_dict is not None:
            save_json(results)
            matrix = gimme_metrics(results)
            accuracy_score, precision_score, _, _, f2_score, _ = show_metrics(matrix[0][0], matrix[0][1], matrix[1][0])

            print(f"Trial {trial.number}: Calculated F½@0.5 = {f2_score:.4f} & Accuracy = {accuracy_score:.4f}")
        else:
            print(f"❌ Trial {trial.number}: Could not access metrics from results object directly.")
            f2_score = 0.0 # Return 0.0 if metrics cannot be accessed
            accuracy_score = 0.0 # Return 0.0 if metrics cannot be accessed
            precision_score = 0.0 # Return 0.0 if metrics cannot be accessed

        # 4. SAVE RESULTS
        print()
        save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/YOLO/save_optuna/study_name/')
        print()
        save_on_cloud(source=f'/content/{optuna_name}', destination='/content/drive/MyDrive/YOLO/save_optuna/')

        print("\n","="*100)
        return f2_score, accuracy_score, precision_score

    except Exception as e:
        print(f"Trial {trial.number}: An error occurred during validation: {e}")
        # Retornar un valor bajo para indicar que este conjunto de hiperparámetros
        # probablemente no es bueno o causó un error.
        return 0.0, 0.0, 0.0 # Return a tuple matching the number of objectives

In [133]:
import logging
import sys

# Optional: Configure logging for Optuna
logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# Define the storage path for the Optuna study
# Using a local SQLite database file
optuna_name = f"optuna_val_study_{model_num}.db"
db_path = f"sqlite:///{optuna_name}"
study_name = f"validation_{model_num}"

# The study progress is automatically saved to 'optuna_yolov8_f1_study.db'
# in the same directory where you run the script.

In [134]:
# Option for setting max n_trials with GPU resources
# gpu_limit = 1 # Setted at the beginning
n_trials = round(gpu_limit*3600/average_time)

print(f"Starting training for {gpu_limit} hour limit")
print(f"Optuna will run {n_trials}")

Starting training for 0.5 hour limit
Optuna will run 108


In [135]:
import time

# Define manually the number of trials to do (optional)
#n_trials = 5 # You can start with a small number, e.g., 50 or 100

# --- Study Creation ---
# Check if the study already exists. If so, load it; otherwise, create a new one.
# This allows resuming the optimization later.
try:
    # Load the existing study
    study = optuna.load_study(study_name=study_name, storage=db_path)
    print(f"Resuming existing study '{study_name}' from {db_path}")
except KeyError:
    # Create a new study if it doesn't exist
    objective_directions = ["maximize", "maximize", "maximize"]
    study = optuna.create_study(study_name=study_name, storage=db_path, directions=objective_directions)
    print(f"Created a new study '{study_name}' at {db_path}")

# --- Study Execution ---
print(f"Running Optuna multi-objective optimization ({n_trials} trials)...")
# Measuring experiment time
start_time = time.perf_counter_ns()

# Run the optimization
# Increase n_trials for a more exhaustive search.
study.optimize(objective, n_trials=n_trials)

# Stop time measurment
end_time = time.perf_counter_ns()


# --- Study Results Analytics ---
print("\nOptimization finished.")

# In multi-objective optimization, access the Pareto front
print("\nTrials on the Pareto front (representing good trade-offs):")
pareto_trials = study.best_trials # Get the list of trials on the Pareto front

if not pareto_trials:
    print("No trials found on the Pareto front.")
else:
    for i, trial in enumerate(pareto_trials):
        print(f"\n  Pareto Front Trial {i+1} (Trial Number: {trial.number}):")
        # Access objective values using .values (plural)
        print(f"    Objective Values (F2, Accuracy, Precision): {trial.values}")
        # Access hyperparameters using .params
        print(f"    Hyperparameters: {trial.params}")

pareto_trials = study.best_trials

selected_trial = None

if not pareto_trials:
    print("\nWarning: No trials found on the Pareto front. Cannot perform final validation with 'best' params.")
else:
    # --- DECIDE WHICH TRIAL TO SELECT ---
    # Option 1: Simply pick the first trial on the Pareto front
    # selected_trial = pareto_trials[0]
    # print(f"\nSelecting the first trial on the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 2: Pick the trial with the highest value for a specific metric (e.g., F2-score)
    # The metrics are in the order: (F2, Accuracy, Precision) -> index 0 is F2
    best_f2_trial = max(pareto_trials, key=lambda t: t.values[0]) # Use index 0 for F2
    selected_trial = best_f2_trial
    #print(f"\nSelecting the trial with the highest F2-score ({selected_trial.values[0]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 3: Pick the trial with the highest Accuracy (index 1)
    # best_accuracy_trial = max(pareto_trials, key=lambda t: t.values[1]) # Use index 1 for Accuracy
    # selected_trial = best_accuracy_trial
    # print(f"\nSelecting the trial with the highest Accuracy ({selected_trial.values[1]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 4: Pick the trial with the highest Precision (index 2)
    # best_precision_trial = max(pareto_trials, key=lambda t: t.values[2]) # Use index 2 for Precision
    # selected_trial = best_precision_trial
    # print(f"\nSelecting the trial with the highest Precision ({selected_trial.values[2]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # You must uncomment and use ONE of the selection options above.
    # Option 2 (max F2) is provided as the default example below.

print() # Add a blank line for spacing

# The rest of your timing code remains the same
elapsed_time_ns = end_time - start_time
elapsed_time_s = elapsed_time_ns / 1e9
average_time = elapsed_time_s/n_trials

print(f"\n\nElapsed time: {elapsed_time_s:.2f} seconds for {n_trials}")
print(f"Average time per trial: {average_time:.3f} seconds")

Resuming existing study 'validation_e86' from sqlite:///optuna_val_study_e86.db
Running Optuna multi-objective optimization (108 trials)...


5️⃣ Trial 5: Trying conf=0.2418, iou=0.5266
VALIDATION:
Ultralytics 8.3.135 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1481.6±583.9 MB/s, size: 100.1 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.35s/it]


                   all        108       3467      0.655      0.517      0.578      0.238
Speed: 5.1ms preprocess, 25.4ms inference, 0.0ms loss, 2.2ms postprocess per image
Saving runs/detect/val6/predictions.json...
Results saved to runs/detect/val6

✅ JSON file stored in: runs/detect/val6
Total objects detected: 4237.0

Confusion matrix:
[ 46.38% , 18.17% ]
[ 35.45% , 0.00% ]

Metrics:
- Accuracy: 0.464
- Precision: 0.718
- Recall: 0.567
- F1 Score: 0.682
- F½ Score: 0.682
- G-mean: 0.638
Trial 5: Calculated F½@0.5 = 0.6820 & Accuracy = 0.4638



[I 2025-05-15 18:39:58,030] Trial 5 finished with values: [0.6819601582564032, 0.4637715364644796, 0.7184643510054844] and parameters: {'conf': 0.24184496345859852, 'iou': 0.5266177446707352}.


✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/YOLO/save_optuna/study_name/

✅ File copied successfully:
   /content/optuna_val_study_e86.db 
  --> /content/drive/MyDrive/YOLO/save_optuna/



6️⃣ Trial 6: Trying conf=0.2541, iou=0.3982
VALIDATION:
Ultralytics 8.3.135 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1247.3±203.4 MB/s, size: 82.2 KB)


...

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/YOLO/save_optuna/study_name/

✅ File copied successfully:
   /content/optuna_val_study_e86.db 
  --> /content/drive/MyDrive/YOLO/save_optuna/



7️⃣4️⃣ Trial 74: Trying conf=0.1665, iou=0.3850
VALIDATION:
Ultralytics 8.3.135 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1958.3±662.2 MB/s, size: 100.0 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.24s/it]


                   all        108       3467      0.605      0.576      0.585      0.232
WARNING ⚠️ ConfusionMatrix plot failure: 
Speed: 0.2ms preprocess, 29.1ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val75/predictions.json...
Results saved to runs/detect/val75

✅ JSON file stored in: runs/detect/val75
Total objects detected: 4556.0

Confusion matrix:
[ 48.46% , 23.90% ]
[ 27.63% , 0.00% ]

Metrics:
- Accuracy: 0.485
- Precision: 0.670
- Recall: 0.637
- F1 Score: 0.663
- F½ Score: 0.663
- G-mean: 0.653
Trial 74: Calculated F½@0.5 = 0.6629 & Accuracy = 0.4846



[W 2025-05-15 19:05:44,116] Trial 74 failed with parameters: {'conf': 0.16645798372428666, 'iou': 0.3850281277597625} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-132-a74352824d93>", line 50, in objective
    save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/YOLO/save_optuna/study_name/')
  File "<ipython-input-122-5ac26246fda5>", line 39, in save_on_cloud
    shutil.copytree(source, destination, dirs_exist_ok=True)
  File "/usr/lib/python3.11/shutil.py", line 573, in copytree
    return _copytree(entries=entries, src=src, dst=dst, symlinks=symlinks,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/shutil.py", line 509, in _copytree
    copytree(srcobj, dstname, symlinks, ignore

KeyboardInterrupt: 

## Extract results for best hyperparams

In [ ]:
pareto_trials = study.best_trials

selected_trial = None

if not pareto_trials:
    print("\nWarning: No trials found on the Pareto front. Cannot perform final validation with 'best' params.")
else:
    # --- DECIDE WHICH TRIAL TO SELECT ---
    # Option 1: Simply pick the first trial on the Pareto front
    # selected_trial = pareto_trials[0]
    # print(f"\nSelecting the first trial on the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 2: Pick the trial with the highest value for a specific metric (e.g., F2-score)
    # The metrics are in the order: (F2, Accuracy, Precision) -> index 0 is F2
    best_f2_trial = max(pareto_trials, key=lambda t: t.values[0]) # Use index 0 for F2
    selected_trial = best_f2_trial
    #print(f"\nSelecting the trial with the highest F2-score ({selected_trial.values[0]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 3: Pick the trial with the highest Accuracy (index 1)
    # best_accuracy_trial = max(pareto_trials, key=lambda t: t.values[1]) # Use index 1 for Accuracy
    # selected_trial = best_accuracy_trial
    # print(f"\nSelecting the trial with the highest Accuracy ({selected_trial.values[1]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # Option 4: Pick the trial with the highest Precision (index 2)
    # best_precision_trial = max(pareto_trials, key=lambda t: t.values[2]) # Use index 2 for Precision
    # selected_trial = best_precision_trial
    # print(f"\nSelecting the trial with the highest Precision ({selected_trial.values[2]:.4f}) from the Pareto front (Trial Number: {selected_trial.number}) for final validation.")

    # You must uncomment and use ONE of the selection options above.
    # Option 2 (max F2) is provided as the default example below.


In [ ]:
# If a trial was successfully selected:
if selected_trial:
    # Get the hyperparameters from the selected trial
    best_conf = selected_trial.params['conf']
    best_iou = selected_trial.params['iou']

    print(f"Using selected best hyperparameters: conf={best_conf:.4f}, iou={best_iou:.4f}")

    # Validate the model with the selected best parameters
    # Ensure 'model' and 'data' variables are accessible in this scope
    print("\nPerforming final validation with selected best parameters:")
    results = model.val(
        data=data, # Use the correct data path
        batch=64,
        conf=best_conf,
        iou=best_iou,
        verbose=True,
        save_json=True
    )

    # You might want to process and display the results of this final validation run
    # similar to how you did in the objective function.
    if hasattr(results, 'results_dict') and results.results_dict is not None:
        # save_json(results) # You might want to save these specific results too
        matrix = gimme_metrics(results) # Ensure gimme_metrics is accessible
        final_accuracy_score, final_precision_score, _, _, final_f2_score, _ = show_metrics(matrix[0][0], matrix[0][1], matrix[1][0]) # Ensure show_metrics is accessible

        print(f"\nFinal Validation Results:")
        print(f"  F½@0.5: {final_f2_score:.4f}")
        print(f"  Accuracy: {final_accuracy_score:.4f}")
        print(f"  Precision: {final_precision_score:.4f}")
    else:
        print("\nCould not access metrics from the final validation results object.")

Using selected best hyperparameters: conf=0.2615, iou=0.3267

Performing final validation with selected best parameters:
Ultralytics 8.3.135 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 112MB/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.2±464.0 MB/s, size: 84.8 KB)



val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<00:00, 959.92it/s]

val: New cache created: /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/valid/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.46s/it]


                   all        108       3467      0.688      0.482      0.574      0.239
Speed: 4.7ms preprocess, 21.2ms inference, 0.0ms loss, 4.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val
Total objects detected: 4069.0

Confusion matrix:
[ 44.90% , 14.79% ]
[ 40.30% , 0.00% ]

Metrics:
- Accuracy: 0.449
- Precision: 0.752
- Recall: 0.527
- F1 Score: 0.693
- F½ Score: 0.693
- G-mean: 0.630

Final Validation Results:
  F½@0.5: 0.6929
  Accuracy: 0.4490
  Precision: 0.7522


In [ ]:
print(best_conf, best_iou)

0.26145314630762906 0.32669144121717214


## Save all results

Guarda una copia de seguridad en carpeta del usuario como respaldo

In [141]:
# Store model weights and metrics
save_on_cloud(source=f'/content/{optuna_name}', destination='/content/drive/MyDrive/save_optuna/')

✅ File copied successfully:
   /content/optuna_val_study_e86.db 
  --> /content/drive/MyDrive/save_optuna/


In [142]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save_optuna/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save_optuna/


# Option for loading previous searches

In [ ]:
import optuna

# 2. Ruta al archivo de la base de datos de Optuna.
#    Generalmente es un archivo .db. Usa el prefijo 'sqlite:///' para bases de datos SQLite.
#    Ejemplo: 'sqlite:///mis_experimentos/optuna_results.db'
#    Asegúrate de que la ruta sea correcta para tu sistema.
db_path = "sqlite:///optuna_val_study_e86.db" # <-- ¡CAMBIA ESTO!

# --- CARGAR EL ESTUDIO DESDE LA BASE DE DATOS ---
try:
    print(f"Intentando cargar el estudio '{study_name}' desde '{db_path}'...")
    study = optuna.load_study(study_name=study_name, storage=db_path)
    print("Estudio cargado exitosamente.")

except Exception as e:
    print(f"\nOcurrió un error inesperado al cargar el estudio: {e}")
    print("Verifica la ruta de la base de datos y el formato del 'storage' string (ej. 'sqlite:///').")
    # Considera salir del script en caso de otros errores de carga
    exit()


Intentando cargar el estudio 'validation_e86' desde 'sqlite:///optuna_val_study_e86.db'...
Estudio cargado exitosamente.
